# 05 Train Main Embedder

Цель этапа: обучить event-only Transformer encoder на prefix-последовательностях.

Objective v0:

- masked event modeling;
- next-event auxiliary head;
- validation metrics: `loss`, `HitRate@10`, `MRR@10`.

Артефакты:

- `artifacts/checkpoints/checkpoint_last.pt`
- `artifacts/checkpoints/checkpoint_best.pt`
- `artifacts/manifests/train_run_state.json`
- W&B run в project `bert4rec-events-embedder`.

In [1]:
from pathlib import Path
import copy
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/root/projects/bert4rec')

## Proxy / W&B

Локальные env-переменные читаются из `.env.local`. Файл добавлен в `.gitignore`, поэтому прокси не должен попадать в артефакты проекта.

In [2]:
def load_local_env(path: Path) -> None:
    if not path.exists():
        print(f"No local env file: {path}")
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")


load_local_env(PROJECT_ROOT / ".env.local")

print("WANDB_MODE:", os.environ.get("WANDB_MODE"))
print("HTTP_PROXY set:", bool(os.environ.get("HTTP_PROXY") or os.environ.get("http_proxy")))
print("HTTPS_PROXY set:", bool(os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")))
print("ALL_PROXY set:", bool(os.environ.get("ALL_PROXY") or os.environ.get("all_proxy")))

WANDB_MODE: online
HTTP_PROXY set: True
HTTPS_PROXY set: True
ALL_PROXY set: True


In [3]:
from src.training import load_train_config, train_main_embedder

base_config = load_train_config(PROJECT_ROOT / "configs" / "train.yaml")
base_config

TrainConfig(paths={'train_prefixes': 'data/processed/train_prefixes.parquet', 'valid_prefixes': 'data/processed/valid_prefixes.parquet', 'test_prefixes': 'data/processed/test_prefixes.parquet', 'event_vocab': 'artifacts/vocab/event_token_vocab.json', 'checkpoint_dir': 'artifacts/checkpoints', 'run_state_path': 'artifacts/manifests/train_run_state.json'}, wandb={'project': 'bert4rec-events-embedder', 'entity': None, 'mode': 'online', 'run_name': 'event-transformer-v0', 'group': 'main-embedder', 'tags': ['train', 'transformer', 'event-only', 'mlm', 'next-event']}, model={'d_model': 256, 'n_heads': 8, 'n_layers': 6, 'ffn_dim': 1024, 'dropout': 0.1, 'max_seq_len': 151, 'time_gap_vocab_size': 10, 'session_vocab_size': 2}, train={'seed': 42, 'device': 'auto', 'batch_size': 64, 'eval_batch_size': 128, 'num_workers': 2, 'epochs': 10, 'max_steps': None, 'validate_every_steps': 500, 'log_every_steps': 50, 'checkpoint_every_steps': 1000, 'gradient_accumulation_steps': 4, 'mixed_precision': True, 

## Control Cell

Сначала можно выполнить `SMOKE_RUN = True`: маленький subset, offline W&B, несколько шагов, проверка checkpoint/resume. Для настоящего запуска поставьте `SMOKE_RUN = False`.

Если запускаете полноценный online run, убедитесь, что W&B login уже сделан и proxy env выставлен, если нужен.

In [4]:
SMOKE_RUN = False
RESUME = True

config = copy.deepcopy(base_config)

if SMOKE_RUN:
    config.wandb["mode"] = "offline"
    config.wandb["run_name"] = "event-transformer-v0-smoke"
    config.paths["checkpoint_dir"] = "artifacts/checkpoints_train_smoke"
    config.paths["run_state_path"] = "artifacts/manifests/train_notebook_smoke_run_state.json"
    config.model["d_model"] = 32
    config.model["n_heads"] = 4
    config.model["n_layers"] = 1
    config.model["ffn_dim"] = 128
    config.train["dry_run_train_rows"] = 512
    config.train["dry_run_valid_rows"] = 128
    config.train["batch_size"] = 16
    config.train["eval_batch_size"] = 32
    config.train["num_workers"] = 0
    config.train["epochs"] = 1
    config.train["max_steps"] = 2
    config.train["validate_every_steps"] = 1
    config.train["log_every_steps"] = 1
    config.train["checkpoint_every_steps"] = 5

print("SMOKE_RUN:", SMOKE_RUN)
print("wandb mode:", config.wandb["mode"])
print("run name:", config.wandb["run_name"])
print("max_steps:", config.train["max_steps"])
print("dry rows:", config.train["dry_run_train_rows"], config.train["dry_run_valid_rows"])

SMOKE_RUN: False
wandb mode: online
run name: event-transformer-v0
max_steps: None
dry rows: None None


In [5]:
result = train_main_embedder(
    config=config,
    project_root=PROJECT_ROOT,
    resume=RESUME,
)

result

/root/projects/bert4rec/src/model_event_encoder.py:48: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.n_layers)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: bolevard (bolevard-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇█████
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
train/grad_norm,▂█▆▅▄▃▂▃▂▃▃▁▂▂▃▂▂▂▂▃▁▂▂▂▃▂▂▂▂▁▂▂▁▂▂▂▂▁▂▁
train/loss_mlm,██▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss_next,█▅▄▄▃▂▂▂▂▂▁▂▂▂▂▂▁▂▁▂▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss_total,█▆▅▅▄▃▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,▁▃▄▆▇███████▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂
train/tokens_seen,▇▆▃▆▆▆▂▁▅▃▇▃▃▃▄▅▅▆▃▃▂▃▁▆▇▄▅▅▇▄▃█▄▂█▃▁▄▂▅
valid/hit_at_10,▁████████
valid/loss_mlm,█▃▂▁▁▁▁▁▁
+3,...


{'global_step': 3780,
 'best_metric': 0.786122858864176,
 'checkpoint_last': '/root/projects/bert4rec/artifacts/checkpoints/checkpoint_last.pt',
 'checkpoint_best': '/root/projects/bert4rec/artifacts/checkpoints/checkpoint_best.pt',
 'final_metrics': {'valid/loss_total': 1.257758992146281,
  'valid/loss_mlm': 0.518819250212438,
  'valid/loss_next': 1.4778794816926237,
  'valid/hit_at_10': 0.9295353037056039,
  'valid/mrr_at_10': 0.7881734890538138},
 'train_rows': 96657,
 'valid_rows': 21046}

Для полноценного запуска:

```python
SMOKE_RUN = False
RESUME = True
```

Первые метрики, на которые смотрим в W&B:

- `train/loss_total`, `train/loss_mlm`, `train/loss_next`
- `valid/loss_total`, `valid/hit_at_10`, `valid/mrr_at_10`
- `train/grad_norm`, `train/lr`
- наличие `checkpoint_last.pt` и `checkpoint_best.pt`.